In [ ]:
# Task 4: Optimize Portfolio Based on Forecast
# Modern Portfolio Theory implementation using forecasted TSLA returns

import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import pickle
import json

# Import custom modules
from portfolio_optimizer import PortfolioOptimizer
from risk_metrics import PortfolioRiskMetrics
from forecast_generator import ForecastGenerator  # For loading forecast

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Load configurations
with open('../config/portfolio_config.yaml', 'r') as f:
    portfolio_config = yaml.safe_load(f)

print("Configuration loaded successfully.")

### Prepare Data for Portfolio Optimization

In [ ]:
print("=" * 80)
print("TASK 4: PORTFOLIO OPTIMIZATION")
print("=" * 80)

print("\n1. LOADING DATA AND FORECASTS...")

# Load historical returns for all assets
returns_path = '../data/processed/daily_returns.csv'
all_returns = pd.read_csv(returns_path, index_col=0, parse_dates=True)

print(f"Loaded returns data: {all_returns.shape}")
print(f"Date range: {all_returns.index[0]} to {all_returns.index[-1]}")
print(f"Assets: {list(all_returns.columns)}")

# Load TSLA forecast from Task 3
print("\nLoading TSLA forecast from Task 3...")
forecast_gen = ForecastGenerator(None)  # Dummy generator for loading
tsla_forecast = forecast_gen.load_forecast_results('../models/tsla_12month_forecast.pkl')

# Extract forecasted returns (convert price forecast to returns)
tsla_price_forecast = tsla_forecast['point_forecast']
tsla_forecast_returns = np.diff(tsla_price_forecast) / tsla_price_forecast[:-1]

print(f"TSLA forecast loaded: {len(tsla_price_forecast)} price points")
print(f"Forecast returns calculated: {len(tsla_forecast_returns)} return points")
print(f"Average forecast return: {np.mean(tsla_forecast_returns)*100:.3f}%")

# Prepare expected returns vector
# For TSLA: Use forecasted average return
# For BND and SPY: Use historical average return

tsla_expected_return = (1 + np.mean(tsla_forecast_returns)) ** 252 - 1  # Annualize
bnd_expected_return = (1 + all_returns['BND'].mean()) ** 252 - 1
spy_expected_return = (1 + all_returns['SPY'].mean()) ** 252 - 1

print(f"\nExpected Annual Returns:")
print(f"  TSLA (forecast-based): {tsla_expected_return:.2%}")
print(f"  BND (historical): {bnd_expected_return:.2%}")
print(f"  SPY (historical): {spy_expected_return:.2%}")

# Create combined returns DataFrame for optimization
# We'll use historical returns for covariance calculation
# But replace TSLA returns with forecast-based expectation

optimization_returns = all_returns.copy()

# Adjust TSLA returns to match forecast expectation (scaling factor)
current_tsla_mean = all_returns['TSLA'].mean()
forecast_tsla_mean = np.mean(tsla_forecast_returns)

if current_tsla_mean != 0:
    scale_factor = forecast_tsla_mean / current_tsla_mean
    # Apply scaling to TSLA returns (preserving covariance structure)
    optimization_returns['TSLA'] = all_returns['TSLA'] * scale_factor

print(f"\nTSLA return scaling factor: {scale_factor:.3f}")
print(f"Adjusted TSLA mean return: {optimization_returns['TSLA'].mean()*100:.3f}%")
print(f"Target forecast mean: {forecast_tsla_mean*100:.3f}%")

### Initialize Portfolio Optimizer

In [ ]:
print("\n" + "=" * 80)
print("2. INITIALIZING PORTFOLIO OPTIMIZER")
print("=" * 80)

# Initialize portfolio optimizer
risk_free_rate = portfolio_config['optimization']['risk_free_rate']
optimizer = PortfolioOptimizer(optimization_returns, risk_free_rate)

print(f"Portfolio Optimizer initialized with:")
print(f"  Assets: {optimizer.assets}")
print(f"  Risk-free rate: {risk_free_rate:.2%}")
print(f"  Number of assets: {optimizer.n_assets}")

# Display asset statistics
print(f"\nAsset Statistics (Annualized):")
print("-" * 40)

for asset in optimizer.assets:
    exp_return = optimizer.expected_returns[asset]
    volatility = np.sqrt(optimizer.covariance_matrix.loc[asset, asset])
    sharpe = (exp_return - risk_free_rate) / volatility if volatility > 0 else 0
    
    print(f"{asset}:")
    print(f"  Expected Return: {exp_return:.2%}")
    print(f"  Volatility: {volatility:.2%}")
    print(f"  Sharpe Ratio: {sharpe:.3f}")

# Display correlation matrix
print(f"\nCorrelation Matrix:")
print(optimizer.correlation_matrix.round(3))

# Visualize correlation matrix
corr_fig_path = '../figures/task4/correlation_matrix.png'
optimizer.plot_correlation_heatmap(save_path=corr_fig_path)

### Generate Random Portfolios

In [ ]:
print("\n" + "=" * 80)
print("3. GENERATING RANDOM PORTFOLIOS")
print("=" * 80)

# Generate random portfolios
n_portfolios = portfolio_config['optimization']['n_portfolios']
random_portfolios = optimizer.generate_random_portfolios(n_portfolios=n_portfolios)

print(f"Generated {len(random_portfolios)} random portfolios")
print(f"Return range: {random_portfolios['return'].min():.2%} to {random_portfolios['return'].max():.2%}")
print(f"Risk range: {random_portfolios['risk'].min():.2%} to {random_portfolios['risk'].max():.2%}")
print(f"Sharpe ratio range: {random_portfolios['sharpe_ratio'].min():.3f} to {random_portfolios['sharpe_ratio'].max():.3f}")

### Optimize Key Portfolios

In [ ]:
print("\n" + "=" * 80)
print("4. OPTIMIZING KEY PORTFOLIOS")
print("=" * 80)

# 4.1 Maximum Sharpe Ratio Portfolio
print("\n4.1 Maximum Sharpe Ratio (Tangency) Portfolio:")
max_sharpe_portfolio = optimizer.optimize_max_sharpe()
print(f"  Expected Return: {max_sharpe_portfolio['expected_return']:.2%}")
print(f"  Volatility: {max_sharpe_portfolio['volatility']:.2%}")
print(f"  Sharpe Ratio: {max_sharpe_portfolio['sharpe_ratio']:.3f}")
print(f"  Value at Risk (95%): {max_sharpe_portfolio['var_95']:.2%}")

print("  Portfolio Weights:")
for i, asset in enumerate(optimizer.assets):
    weight = max_sharpe_portfolio['weights'][i]
    if weight > 0.001:
        print(f"    {asset}: {weight:.1%}")

# 4.2 Minimum Volatility Portfolio
print("\n4.2 Minimum Volatility Portfolio:")
min_vol_portfolio = optimizer.optimize_min_volatility()
print(f"  Expected Return: {min_vol_portfolio['expected_return']:.2%}")
print(f"  Volatility: {min_vol_portfolio['volatility']:.2%}")
print(f"  Sharpe Ratio: {min_vol_portfolio['sharpe_ratio']:.3f}")

print("  Portfolio Weights:")
for i, asset in enumerate(optimizer.assets):
    weight = min_vol_portfolio['weights'][i]
    if weight > 0.001:
        print(f"    {asset}: {weight:.1%}")

# 4.3 Equal Weight Portfolio
print("\n4.3 Equal Weight Portfolio:")
equal_weight_portfolio = optimizer.calculate_equal_weight_portfolio()
print(f"  Expected Return: {equal_weight_portfolio['expected_return']:.2%}")
print(f"  Volatility: {equal_weight_portfolio['volatility']:.2%}")
print(f"  Sharpe Ratio: {equal_weight_portfolio['sharpe_ratio']:.3f}")

print("  Portfolio Weights:")
for i, asset in enumerate(optimizer.assets):
    weight = equal_weight_portfolio['weights'][i]
    print(f"    {asset}: {weight:.1%}")

# 4.4 Target Return Portfolios
print("\n4.4 Target Return Portfolios:")
target_returns = portfolio_config['optimization']['target_returns']

for target in target_returns:
    try:
        target_portfolio = optimizer.optimize_target_return(target)
        print(f"  Target Return {target:.0%}:")
        print(f"    Achieved Return: {target_portfolio['expected_return']:.2%}")
        print(f"    Volatility: {target_portfolio['volatility']:.2%}")
        print(f"    Sharpe Ratio: {target_portfolio['sharpe_ratio']:.3f}")
    except Exception as e:
        print(f"  Target Return {target:.0%}: Optimization failed - {str(e)}")

### Generate Efficient Frontier

In [ ]:
print("\n" + "=" * 80)
print("5. GENERATING EFFICIENT FRONTIER")
print("=" * 80)

# Generate efficient frontier
n_points = portfolio_config['efficient_frontier']['n_points']
efficient_frontier = optimizer.generate_efficient_frontier(n_points=n_points)

print(f"Generated efficient frontier with {len(efficient_frontier)} points")
print(f"Minimum risk portfolio:")
print(f"  Return: {efficient_frontier['expected_return'].min():.2%}")
print(f"  Risk: {efficient_frontier['volatility'].min():.2%}")

print(f"\nMaximum return portfolio:")
print(f"  Return: {efficient_frontier['expected_return'].max():.2%}")
print(f"  Risk: {efficient_frontier['volatility'].max():.2%}")

# Find maximum Sharpe portfolio on efficient frontier
max_sharpe_idx = efficient_frontier['sharpe_ratio'].idxmax()
max_sharpe_ef = efficient_frontier.loc[max_sharpe_idx]

print(f"\nMaximum Sharpe portfolio on efficient frontier:")
print(f"  Return: {max_sharpe_ef['expected_return']:.2%}")
print(f"  Risk: {max_sharpe_ef['volatility']:.2%}")
print(f"  Sharpe Ratio: {max_sharpe_ef['sharpe_ratio']:.3f}")

# Visualize efficient frontier
ef_fig_path = '../figures/task4/efficient_frontier.png'
optimizer.plot_efficient_frontier(save_path=ef_fig_path)

# Visualize portfolio composition
composition_fig_path = '../figures/task4/portfolio_composition.png'
optimizer.plot_portfolio_composition(save_path=composition_fig_path)

### Calculate Advanced Risk Metrics

In [ ]:
print("\n" + "=" * 80)
print("6. CALCULATING ADVANCED RISK METRICS")
print("=" * 80)

# Calculate returns for key portfolios (simulated)
n_simulations = 1000
simulated_days = 252  # One year of daily returns

print("\nSimulating portfolio returns for risk metric calculation...")

# Function to simulate portfolio returns
def simulate_portfolio_returns(weights, cov_matrix, expected_returns, n_days=252, n_sims=1000):
    """Simulate portfolio returns using multivariate normal distribution"""
    portfolio_returns = []
    
    # Calculate portfolio expected return and covariance
    port_return = np.sum(expected_returns * weights) / 252  # Daily expected return
    port_cov = np.dot(weights.T, np.dot(cov_matrix / 252, weights))  # Daily covariance
    
    # Simulate returns
    for _ in range(n_sims):
        sim_returns = np.random.normal(port_return, np.sqrt(port_cov), n_days)
        portfolio_returns.append(sim_returns)
    
    return np.array(portfolio_returns)

# Calculate metrics for each key portfolio
for port_name, port_metrics in optimizer.key_portfolios.items():
    if port_name in ['max_sharpe', 'min_volatility', 'equal_weight']:
        print(f"\n{port_name.replace('_', ' ').title()} Portfolio - Advanced Risk Metrics:")
        
        # Simulate returns
        weights = port_metrics['weights']
        sim_returns = simulate_portfolio_returns(
            weights, 
            optimizer.covariance_matrix.values,
            optimizer.expected_returns.values,
            n_days=simulated_days,
            n_sims=n_simulations
        )
        
        # Calculate all metrics for first simulation (as example)
        example_returns = sim_returns[0]
        
        # Simulate price series from returns
        initial_price = 100  # Starting with $100
        prices = initial_price * np.cumprod(1 + example_returns)
        
        # Calculate advanced metrics
        risk_metrics = PortfolioRiskMetrics.calculate_all_metrics(
            portfolio_returns=example_returns,
            portfolio_prices=prices,
            risk_free_rate=risk_free_rate
        )
        
        # Display key metrics
        print(f"  Average Daily Return: {risk_metrics['avg_return']*100:.3f}%")
        print(f"  Daily Volatility: {risk_metrics['volatility']*100:.3f}%")
        print(f"  Sharpe Ratio: {risk_metrics['sharpe_ratio']:.3f}")
        print(f"  Value at Risk (95%): {risk_metrics['var_95']*100:.3f}%")
        print(f"  Conditional VaR (95%): {risk_metrics['cvar_95']*100:.3f}%")
        print(f"  Maximum Drawdown: {risk_metrics['max_drawdown']*100:.2f}%")
        print(f"  Sortino Ratio: {risk_metrics['sortino_ratio']:.3f}")
        print(f"  Calmar Ratio: {risk_metrics['calmar_ratio']:.3f}")

### Portfolio Recommendation and Justification

In [ ]:
print("\n" + "=" * 80)
print("7. PORTFOLIO RECOMMENDATION AND JUSTIFICATION")
print("=" * 80)

# Compare key portfolios
print("\nPORTFOLIO COMPARISON:")
print("-" * 60)
print(f"{'Portfolio':<20} {'Return':>10} {'Risk':>10} {'Sharpe':>10} {'VaR (95%)':>12}")
print("-" * 60)

for port_name in ['max_sharpe', 'min_volatility', 'equal_weight']:
    if port_name in optimizer.key_portfolios:
        port = optimizer.key_portfolios[port_name]
        display_name = port_name.replace('_', ' ').title()
        
        print(f"{display_name:<20} {port['expected_return']:>9.2%} "
              f"{port['volatility']:>9.2%} {port['sharpe_ratio']:>9.3f} "
              f"{port['var_95']:>11.2%}")

print("-" * 60)

# Make recommendation
max_sharpe = optimizer.key_portfolios['max_sharpe']
min_vol = optimizer.key_portfolios['min_volatility']

print(f"\n🎯 PRIMARY RECOMMENDATION: Maximum Sharpe Ratio Portfolio")
print("=" * 60)

print(f"\nRATIONALE:")
print(f"1. Superior Risk-Adjusted Returns:")
print(f"   • Sharpe Ratio: {max_sharpe['sharpe_ratio']:.3f} (vs {min_vol['sharpe_ratio']:.3f} for Min Vol)")
print(f"   • Best return per unit of risk taken")

print(f"\n2. Optimal Diversification:")
print(f"   • Diversification Ratio: {max_sharpe['diversification_ratio']:.2f}x")
print(f"   • Combines growth (TSLA) with stability (BND) effectively")

print(f"\n3. Realistic Return Expectations:")
print(f"   • Expected Return: {max_sharpe['expected_return']:.2%}")
print(f"   • Based on forecast-informed TSLA expectations")
print(f"   • Balanced with historical BND/SPY performance")

print(f"\n4. Risk Management:")
print(f"   • Volatility: {max_sharpe['volatility']:.2%} (manageable)")
print(f"   • Value at Risk (95%): {max_sharpe['var_95']:.2%}")
print(f"   • TSLA exposure controlled via optimization")

print(f"\nRECOMMENDED ALLOCATION:")
print("=" * 60)

for i, asset in enumerate(optimizer.assets):
    weight = max_sharpe['weights'][i]
    if weight > 0.001:
        contribution = weight * optimizer.expected_returns[asset]
        risk_contribution = weight * np.sqrt(optimizer.covariance_matrix.loc[asset, asset])
        
        print(f"\n{asset}:")
        print(f"  Allocation: {weight:.1%}")
        print(f"  Contribution to Return: {contribution:.3%}")
        print(f"  Contribution to Risk: {risk_contribution:.3%}")
        
        # Investment rationale
        if asset == 'TSLA':
            print(f"  Rationale: Growth engine with forecasted {tsla_expected_return:.2%} return")
        elif asset == 'BND':
            print(f"  Rationale: Stability anchor with {bnd_expected_return:.2%} return")
        elif asset == 'SPY':
            print(f"  Rationale: Market diversification with {spy_expected_return:.2%} return")

print(f"\nPORTFOLIO CHARACTERISTICS:")
print("=" * 60)
print(f"Expected Annual Return: {max_sharpe['expected_return']:.2%}")
print(f"Annual Volatility: {max_sharpe['volatility']:.2%}")
print(f"Sharpe Ratio: {max_sharpe['sharpe_ratio']:.3f}")
print(f"Value at Risk (95%): {max_sharpe['var_95']:.2%}")
print(f"Maximum Drawdown (Est.): {max_sharpe['max_drawdown_est']:.2%}")
print(f"Diversification Benefit: {max_sharpe['diversification_ratio']:.2f}x")

print(f"\nIMPLEMENTATION GUIDANCE:")
print("=" * 60)
print(f"1. Initial Investment: Allocate according to recommended weights")
print(f"2. Rebalancing: Quarterly rebalancing to maintain target weights")
print(f"3. Monitoring: Track correlation changes between assets")
print(f"4. Risk Controls: Implement 10% stop-loss on TSLA position")
print(f"5. Review: Update portfolio monthly with new forecast data")

print(f"\nALTERNATIVE SCENARIOS:")
print("=" * 60)
print(f"For Risk-Averse Investors: Minimum Volatility Portfolio")
print(f"  • Return: {min_vol['expected_return']:.2%}")
print(f"  • Risk: {min_vol['volatility']:.2%}")
print(f"  • Recommended for capital preservation objectives")

print(f"\nFor Passive Investors: Equal Weight Portfolio")
print(f"  • Return: {equal_weight_portfolio['expected_return']:.2%}")
print(f"  • Risk: {equal_weight_portfolio['volatility']:.2%}")
print(f"  • Maximum diversification with minimal maintenance")

### Save Optimization Results

In [ ]:
print("\n" + "=" * 80)
print("8. SAVING OPTIMIZATION RESULTS")
print("=" * 80)

# Save optimization results
results_path = '../models/portfolio_optimization_results.pkl'
optimizer.save_optimization_results(results_path)

print(f"Optimization results saved to: {results_path}")

# Save portfolio weights for Task 5 backtesting
portfolio_weights = {
    'max_sharpe': max_sharpe['weights'],
    'min_volatility': min_vol['weights'],
    'equal_weight': equal_weight_portfolio['weights'],
    'assets': optimizer.assets,
    'risk_free_rate': risk_free_rate,
    'generation_date': pd.Timestamp.now()
}

weights_path = '../models/portfolio_weights.pkl'
with open(weights_path, 'wb') as f:
    pickle.dump(portfolio_weights, f)

print(f"Portfolio weights saved to: {weights_path}")

# Create CSV file with portfolio allocations
allocations_df = pd.DataFrame({
    'Asset': optimizer.assets,
    'Max_Sharpe_Weight': max_sharpe['weights'],
    'Min_Volatility_Weight': min_vol['weights'],
    'Equal_Weight': equal_weight_portfolio['weights']
})

allocations_path = '../data/processed/portfolio_allocations.csv'
allocations_df.to_csv(allocations_path, index=False)
print(f"Portfolio allocations saved to: {allocations_path}")